# DAPO Loss

**DAPO: Decoupled Clip and Dynamic sAmpling Policy Optimization**

来自 ByteDance Seed & 清华大学 AIR 实验室 (arXiv:2503.14476)

DAPO 在 GRPO 基础上提出了四项关键改进：
1. **Clip-Higher（解耦裁剪）**：对上下界使用不同的裁剪阈值，鼓励探索低概率 token
2. **Dynamic Sampling（动态采样）**：过滤全对/全错的样本组，保证梯度有效性
3. **Token-Level Loss（token 级别损失）**：用所有 token 总数归一化，而非先对每序列取均值
4. **Overlong Reward Shaping（超长响应惩罚）**：对截断响应施加长度惩罚，减少奖励噪声

## DAPO 目标函数

$$
\mathcal{J}_{\text{DAPO}}(\theta) = \mathbb{E}_{(q,a)\sim\mathcal{D},\{o_i\}_{i=1}^G\sim\pi_{\theta_{\text{old}}}(\cdot|q)}
\frac{1}{\sum_{i=1}^G |o_i|} \sum_{i=1}^G \sum_{t=1}^{|o_i|}
\left[ \min\left(r_{i,t}(\theta)\hat{A}_{i,t},\ \text{clip}\left(r_{i,t}(\theta), 1-\varepsilon_{\text{low}}, 1+\varepsilon_{\text{high}}\right)\hat{A}_{i,t}\right) \right]
$$

**s.t.** $0 < |\{o_i \mid \text{is\_equivalent}(a, o_i)\}| < G$ （动态采样约束：不允许全对或全错的组）

其中：
$$
r_{i,t}(\theta) = \frac{\pi_\theta(o_{i,t} \mid q, o_{i,<t})}{\pi_{\theta_{\text{old}}}(o_{i,t} \mid q, o_{i,<t})}
$$

$$
\hat{A}_{i,t} = \frac{R_i - \text{mean}(\{R_j\}_{j=1}^G)}{\text{std}(\{R_j\}_{j=1}^G)}
$$

## 关键技术 1：Clip-Higher（解耦上下裁剪阈值）

GRPO 使用对称裁剪 $[1-\varepsilon, 1+\varepsilon]$，而 DAPO 解耦上下界：

- $\varepsilon_{\text{low}}$：下界（通常较小，如 0.2），防止低估概率的 token 被过度压制
- $\varepsilon_{\text{high}}$：上界（通常较大，如 0.28），**放宽**低概率 token 的上限，鼓励探索

**动机**：如果 $\pi_{\theta_{\text{old}}}(o_i|q) = 0.01$，在 $\varepsilon=0.2$ 下，概率最多能增加到 $0.012$；
而高概率 token（如 $0.9$）却可以增加到 $1.08$。这不利于低概率探索 token 的学习。
通过增大 $\varepsilon_{\text{high}}$ 可以解决这个问题，避免熵坍缩（entropy collapse）。

## 关键技术 2：Token-Level Loss（token 级别归一化）

| 方法 | 归一化方式 |
|------|----------|
| GRPO | $\frac{1}{G}\sum_i \frac{1}{|o_i|} \sum_t$（先对每条序列取均值，再对组取均值） |
| DAPO | $\frac{1}{\sum_i |o_i|} \sum_i \sum_t$（对所有 token 总数归一化） |

DAPO 的 token 级别损失对长序列赋予更大权重，有利于长链式推理的学习。

## 关键技术 3：Dynamic Sampling（动态采样）

若组内所有输出都正确（奖励全为 1），则 $\hat{A}_i=0$，梯度归零，浪费训练资源。
若组内所有输出都错误，同样如此。

DAPO 动态过滤这些无效样本，持续采样直到批次填满
样本（即组内既有对也有错）。

## 关键技术 4：Overlong Reward Shaping（超长响应惩罚）

$$
R_{\text{length}}(y) = \begin{cases}
0 & |y| \leq L_{\text{max}} \\
\frac{L_{\text{max}} - L_{\text{cache}} - |y|}{L_{\text{cache}}} & L_{\text{max}} < |y| \leq L_{\text{max}} + L_{\text{cache}} \\
-1 & |y| > L_{\text{max}} + L_{\text{cache}}
\end{cases}
$$

对截断的超长响应施加惩罚，减少奖励噪声。注意：DAPO **不使用 KL 散度惩罚项**。

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

## 优势函数（与 GRPO 相同）

In [ ]:
def grpo_advantage(rewards):
    """
    计算组相对优势（Group Relative Advantage）
    
    与 GRPO 相同，将组内奖励标准化为均值为 0、方差为 1 的分布：
        A_i = (R_i - mean(R)) / (std(R) + epsilon)
    
    优势 > 0 表示该响应优于组内平均，鼓励模型提升该响应概率
    优势 < 0 表示该响应劣于组内平均，鼓励模型降低该响应概率
    
    Args:
        rewards: Tensor[G]，组内 G 条响应的奖励值
    Returns:
        A: Tensor[G]，归一化优势值
    """
    epsilon = 1e-5  # 防止除以零
    A = (rewards - rewards.mean()) / (rewards.std() + epsilon)
    return A

## Overlong Reward Shaping（超长响应惩罚）

In [ ]:
def overlong_reward_shaping(response_len, L_max, L_cache):
    """
    超长响应惩罚（Soft Overlong Punishment）
    
    当响应长度超过 L_max 时，施加渐进式惩罚：
    - 响应长度 <= L_max：不惩罚
    - L_max < 长度 <= L_max + L_cache：线性惩罚 [0, -1]
    - 长度 > L_max + L_cache：最大惩罚 -1（截断响应，奖励为 -1）
    
    这个惩罚叠加在原始规则奖励上，告知模型避免过长响应。
    
    Args:
        response_len: 响应长度（token 数量）
        L_max: 最大允许长度
        L_cache: 惩罚过渡区间长度
    Returns:
        length_penalty: 长度惩罚值，范围 [-1, 0]
    """
    if response_len <= L_max:
        return 0.0  # 不超长，无惩罚
    elif response_len <= L_max + L_cache:
        # 线性惩罚：越长惩罚越大
        return (L_max - L_cache - response_len) / L_cache
    else:
        return -1.0  # 超过极限，最大惩罚

# 可视化惩罚函数
L_max, L_cache = 2048, 512
lengths = list(range(1800, 2700))
penalties = [overlong_reward_shaping(l, L_max, L_cache) for l in lengths]

plt.figure(figsize=(10, 4))
plt.plot(lengths, penalties)
plt.axvline(x=L_max, color='r', linestyle='--', label=f'L_max={L_max}')
plt.axvline(x=L_max+L_cache, color='orange', linestyle='--', label=f'L_max+L_cache={L_max+L_cache}')
plt.xlabel('Response Length (tokens)')
plt.ylabel('Length Penalty')
plt.title('Overlong Reward Shaping (Soft Overlong Punishment)')
plt.legend()
plt.grid()
plt.show()

## DAPO Loss 核心实现

In [ ]:
def dapo_loss(
    pi_logprob,       # 当前策略的 log prob，shape: [bs, seq_len]
    pi_old_logprob,   # 旧策略的 log prob，shape: [bs, seq_len]
    rewards,          # 每条响应的奖励，shape: [bs]
    input_len,        # prompt（输入）的 token 数量
    epsilon_low=0.2,  # 下裁剪阈值（与 GRPO 默认值相同）
    epsilon_high=0.28,# 上裁剪阈值（比 epsilon_low 大，鼓励探索）
    is_debug=True
):
    """
    DAPO Loss 实现
    
    与 GRPO 的主要区别：
    1. 解耦上下裁剪：clip(r, 1-ε_low, 1+ε_high)，ε_high > ε_low
    2. Token-level 归一化：用 1/sum(|o_i|) 代替 1/G * 1/|o_i|
    3. 无 KL 散度惩罚项
    
    损失公式：
        L = -(1 / sum_tokens) * sum_i sum_t [ min(r_t * A_i, clip(r_t, 1-ε_low, 1+ε_high) * A_i) ] * mask
    """
    bs, seq_len = pi_logprob.shape
    
    # ============================================================
    # Step 1: 构建输出部分的 mask
    # 只对输出 token 计算损失，输入（prompt）部分不参与训练
    # mask[i, t] = 1 表示第 t 个 token 是输出 token
    # ============================================================
    mask = torch.zeros(bs, seq_len)
    mask[:, input_len:] = 1.0
    
    # 计算有效 token 数量（仅输出部分）
    total_output_tokens = mask.sum()  # DAPO 的关键：用总 token 数归一化，而非每序列单独归一化
    
    # ============================================================
    # Step 2: 计算组相对优势
    # 将组内奖励标准化为 z-score
    # ============================================================
    advantage = grpo_advantage(rewards)  # [bs]
    advantage = advantage.unsqueeze(dim=1)  # [bs] -> [bs, 1]，广播到所有 token
    
    # ============================================================
    # Step 3: 计算重要性采样比率（IS ratio）
    # r_t = π_θ(o_t) / π_θold(o_t)
    # 用 log 差值再取 exp 在数值上更稳定（等价于直接相除）
    # ============================================================
    ratio = torch.exp(pi_logprob - pi_old_logprob)  # [bs, seq_len]
    
    # ============================================================
    # Step 4: DAPO Clip-Higher 关键操作
    # 解耦上下裁剪阈值：下界用 epsilon_low，上界用 epsilon_high
    # 这样低概率 token 有更大空间增加概率（鼓励探索）
    # ============================================================
    ratio_clip = torch.clamp(ratio, 1 - epsilon_low, 1 + epsilon_high)  # 解耦裁剪
    
    # ============================================================
    # Step 5: 计算策略梯度目标（取 min 保证单调性和稳定性）
    # min(r*A, clip(r)*A) 确保对正优势不过度激励、对负优势不过度惩罚
    # ============================================================
    policy_gradient = torch.minimum(ratio * advantage, ratio_clip * advantage)  # [bs, seq_len]
    
    # ============================================================
    # Step 6: 计算最终损失
    # DAPO 的 token-level 归一化：除以所有输出 token 总数（而非先对序列取均值）
    # 这对长序列更公平，鼓励生成更完整的推理链
    # 乘以 -1 将最大化目标转换为最小化损失
    # ============================================================
    loss = -(1.0 / total_output_tokens) * (policy_gradient * mask).sum()
    
    if is_debug:
        print(f'[Rewards]     : {rewards.tolist()}')
        print(f'[Advantage]   : {advantage.squeeze().tolist()}')
        print(f'[Ratio range] : [{ratio[mask==1].min():.4f}, {ratio[mask==1].max():.4f}]')
        print(f'[Total tokens]: {total_output_tokens.item()}')
        print(f'[Loss]        : {loss.item():.6f}')
    
    return loss

## 与 GRPO Loss 对比：Token-Level vs Sequence-Level 归一化

In [ ]:
def grpo_loss_for_compare(pi_logprob, pi_old_logprob, rewards, input_len, len_oi):
    """
    GRPO Loss（用于对比）
    归一化方式：先对每条序列取均值 (1/|o_i|)，再对组取均值 (1/G)
    """
    epsilon = 0.2
    bs, seq_len = pi_logprob.shape
    len_oi_tensor = torch.tensor([len_oi] * bs, dtype=torch.float32)
    mask = torch.zeros(bs, seq_len)
    mask[:, input_len:] = 1.0
    
    ratio = torch.exp(pi_logprob - pi_old_logprob)
    ratio_clip = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)
    advantage = grpo_advantage(rewards).unsqueeze(dim=1)
    policy_gradient = torch.minimum(ratio * advantage, ratio_clip * advantage)
    
    # GRPO 归一化：(1/bs) * sum_i (1/|o_i|) * sum_t
    loss = (-1.0 / bs) * (1.0 / len_oi_tensor.unsqueeze(1)) * policy_gradient * mask
    return loss.sum()


# ----------------------------------------------------------------
# 对比实验：验证 token-level 损失在不均等序列长度时的差异
# ----------------------------------------------------------------
torch.manual_seed(42)
bs, seq_len = 4, 10
input_len = 3
output_len = seq_len - input_len  # = 7

pi_logits = torch.randn(bs, seq_len, 32)
pi_old_logits = pi_logits.clone()  # 初始时新旧策略相同（ratio=1）

token_ids = torch.randint(0, 32, (bs, seq_len))
pi_logprob = F.log_softmax(pi_logits, dim=-1)
pi_logprob = torch.gather(pi_logprob, -1, token_ids.unsqueeze(-1)).squeeze(-1)
pi_old_logprob = pi_logprob.clone()

rewards = torch.tensor([1.0, 0.0, 1.0, 0.0])

loss_dapo = dapo_loss(pi_logprob, pi_old_logprob, rewards, input_len, is_debug=False)
loss_grpo = grpo_loss_for_compare(pi_logprob, pi_old_logprob, rewards, input_len, output_len)

print(f'DAPO Loss (token-level normalization): {loss_dapo.item():.6f}')
print(f'GRPO Loss (sequence-level normalization): {loss_grpo.item():.6f}')
print(f'\n注意：两者核心计算相同，只是归一化方式不同。')
print(f'DAPO 用 1/total_tokens 归一化，GRPO 用 1/G * 1/|o_i| 归一化。')

## Clip-Higher 效果验证：解耦裁剪 vs 对称裁剪

In [ ]:
# 验证 Clip-Higher 对低概率探索 token 的影响
# 模拟一个低概率 token 在正优势情况下的梯度上限

def compute_clipped_upper_bound(prob_old, epsilon_low, epsilon_high):
    """计算正优势下概率的最大增长量"""
    symmetric_upper = prob_old * (1 + epsilon_low)   # 对称裁剪的上界
    decoupled_upper = prob_old * (1 + epsilon_high)  # 解耦裁剪的上界
    return symmetric_upper, decoupled_upper

# 不同初始概率的 token
prob_old_values = [0.01, 0.05, 0.1, 0.3, 0.5, 0.9]
epsilon_low = 0.2
epsilon_high = 0.28  # DAPO 推荐值，比 epsilon_low 大

print(f"{'初始概率':>8} | {'对称裁剪上界(ε=0.2)':>20} | {'解耦裁剪上界(ε_high=0.28)':>22} | {'提升空间增益':>12}")
print('-' * 75)
for p in prob_old_values:
    sym, dec = compute_clipped_upper_bound(p, epsilon_low, epsilon_high)
    gain = (dec - sym) / sym * 100
    print(f"{p:>8.2f} | {sym:>20.4f} | {dec:>22.4f} | {gain:>11.1f}%")

print()
print('结论：对低概率 token（探索 token，如 ε=0.01），解耦裁剪增加了更多提升空间，')
print('      有助于防止 entropy collapse（熵坍缩），保持策略的多样性。')

## Dynamic Sampling（动态采样）模拟

In [ ]:
def dynamic_sampling_filter(rewards_group):
    """
    动态采样过滤器
    
    DAPO 的关键约束：过滤掉全对（全奖励=1）或全错（全奖励=0）的响应组
    
    原因：
    - 全对：所有优势 A_i = 0，梯度为零，浪费计算资源
    - 全错：同上，且可能使训练不稳定
    
    只有组内既有成功又有失败的样本，才能提供有效的学习信号
    
    Args:
        rewards_group: Tensor[G]，组内各响应的奖励（0 或 1）
    Returns:
        is_valid: bool，该组是否有效（有梯度）
    """
    num_correct = (rewards_group == 1).sum().item()
    G = len(rewards_group)
    # 有效条件：至少有 1 个正确，至少有 1 个错误
    is_valid = 0 < num_correct < G
    return is_valid

# 模拟不同场景
test_cases = [
    torch.tensor([1., 0., 1., 0., 0., 0., 0., 0.]),  # 有效：既有对也有错
    torch.tensor([1., 1., 1., 1., 1., 1., 1., 1.]),  # 无效：全对（全局准确率=100%）
    torch.tensor([0., 0., 0., 0., 0., 0., 0., 0.]),  # 无效：全错（全局准确率=0%）
    torch.tensor([1., 1., 1., 1., 0., 0., 0., 0.]),  # 有效：50% 准确率
]

for i, rewards in enumerate(test_cases):
    valid = dynamic_sampling_filter(rewards)
    acc = rewards.mean().item()
    advantage = grpo_advantage(rewards)
    grad_norm = advantage.abs().mean().item()
    print(f'Case {i+1}: accuracy={acc:.1%}, valid={valid}, |gradient|={grad_norm:.4f}')

print()
print('结论：只有 Case 1 和 4 是有效样本，才能产生有意义的梯度。')
print('DAPO 会过滤掉 Case 2 和 3，通过继续采样来替换无效样本组。')

## 完整 DAPO Loss（含 token 级别、批次维度）

In [ ]:
def dapo_loss_full(pi_logprob, pi_old_logprob, rewards, input_len, len_oi,
                   epsilon_low=0.2, epsilon_high=0.28):
    """
    完整的 DAPO Loss（支持批次维度）
    
    与 grpo_loss 相比，三个核心变化：
    1. 解耦裁剪：使用不同的 epsilon_low 和 epsilon_high
    2. Token-level 归一化：用总输出 token 数归一化
    3. 无 KL 散度项
    
    Args:
        pi_logprob: 当前策略 log prob，[bs, seq_len]
        pi_old_logprob: 旧策略 log prob，[bs, seq_len]
        rewards: 各响应奖励，[bs]
        input_len: prompt 长度
        len_oi: 输出序列长度（用于构建 mask）
        epsilon_low: 下裁剪阈值
        epsilon_high: 上裁剪阈值（推荐 > epsilon_low）
    """
    bs, seq_len = pi_logprob.shape
    
    # 构建输出 mask
    mask = torch.zeros(bs, seq_len)
    mask[:, input_len:] = 1.0
    total_output_tokens = mask.sum()  # 所有输出 token 总数
    
    # 优势函数（组相对优势）
    advantage = grpo_advantage(rewards).unsqueeze(1)  # [bs, 1]
    
    # 重要性采样比率
    ratio = torch.exp(pi_logprob - pi_old_logprob)  # [bs, seq_len]
    
    # DAPO Clip-Higher：解耦上下裁剪阈值
    ratio_clip = torch.clamp(ratio, 1 - epsilon_low, 1 + epsilon_high)
    
    # PPO-style min 操作
    policy_gradient = torch.minimum(ratio * advantage, ratio_clip * advantage)
    
    # Token-level 归一化损失（DAPO 关键）
    loss = -(1.0 / total_output_tokens) * (policy_gradient * mask).sum()
    
    return loss


# ======================== 测试 ========================
torch.manual_seed(0)
pi_logits     = torch.randn(3, 7, 32)  # [batch=3, seq=7, vocab=32]
pi_old_logits = torch.randn(3, 7, 32)
pi_ref_logits = torch.randn(3, 7, 32)

pi_logprob_full     = F.log_softmax(pi_logits,     dim=-1)
pi_old_logprob_full = F.log_softmax(pi_old_logits, dim=-1)

token_ids = torch.tensor([[3, 4, 5, 6, 7, 8, 9],
                           [3, 4, 5, 7, 8, 9, 10],
                           [3, 4, 5, 8, 9, 10, 11]])

pi_logprob_full     = torch.gather(pi_logprob_full,     -1, token_ids.unsqueeze(-1)).squeeze(-1)
pi_old_logprob_full = torch.gather(pi_old_logprob_full, -1, token_ids.unsqueeze(-1)).squeeze(-1)

rewards = torch.tensor([1.0, 0.0, 1.0])

loss = dapo_loss_full(pi_logprob_full, pi_old_logprob_full, rewards,
                      input_len=3, len_oi=4, epsilon_low=0.2, epsilon_high=0.28)
print(f'DAPO Loss: {loss.item():.6f}')

## DAPO vs GRPO 总结

| 特性 | GRPO | DAPO |
|------|------|------|
| 裁剪方式 | 对称：$[1-\varepsilon, 1+\varepsilon]$ | 解耦：$[1-\varepsilon_{\text{low}}, 1+\varepsilon_{\text{high}}]$ |
| 损失归一化 | $\frac{1}{G}\sum_i \frac{1}{|o_i|}\sum_t$ | $\frac{1}{\sum_i|o_i|}\sum_i\sum_t$ |
| KL 散度惩罚 | 有（$\beta D_{\text{KL}}$） | 无 |
| 采样策略 | 所有样本 | 过滤全对/全错组 |
| 超长响应 | 无特殊处理 | Soft Overlong Punishment |
| 适用场景 | 一般 RL 对齐 | 长链式推理（Long CoT）|

**核心直觉**：DAPO 专为长链式推理（Long-CoT RL）设计，通过解耦裁剪防止熵坍缩，通过动态采样保证梯度质量，通过 token-level 损失鼓励更长的推理链。